# 01. Pre-Initialization: IDAS 의도 분리 및 요약

**목적**: 원본 상담 대화(`consulting_content`)를 Gemini API로 처리하여 (1) 복수 의도가 포함된 상담을 개별 의도 단위로 분리하고, (2) 각 의도를 노이즈 제거된 요약문으로 변환한다. 이 요약문이 이후 클러스터링 파이프라인의 입력이 된다.

**방법론**: IDAS (Intent-Driven Abstractive Summarization) 아이디어를 활용. LLM이 상담 대화를 읽고 고객의 각 의도를 식별하여 구조화된 JSON으로 출력한다.

**산출물**:
- `data/processed/pre_init_intents.parquet` — 의도 단위로 분리·요약된 전체 데이터
- `checkpoints/pre_init_batch_*.parquet` — 100건 단위 중간 체크포인트

**모델**: `gemini-2.5-flash-lite` (Input $0.10/1M, Output $0.40/1M tokens)

**예상 비용**: ≈ ₩5,200 / ₩50,000 예산

**예상 소요 시간**: ~1.2시간 (Tier 1 기준 300 RPM)

## 0. 환경 설정
Google Drive 마운트, 작업 디렉토리 이동, 필수 라이브러리 설치 및 임포트

In [1]:
# === 환경 설정 ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/5stone_experiment/1_clustering_test')
print(f'작업 디렉토리: {os.getcwd()}')

!pip install -q google-genai

import json
import time
import zipfile
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive
작업 디렉토리: /content/drive/MyDrive/5stone_experiment/1_clustering_test


## 0-1. 경로 설정 (PATH_CONFIG)
모든 파일 경로는 이 딕셔너리를 통해 참조한다. macOS ↔ Linux 간 한글 파일명 인코딩 차이(NFD/NFC)를 `unicodedata.normalize`로 통일한다.

In [2]:
# === 경로 설정 ===
BASE_DIR = Path('/content/drive/MyDrive/5stone_experiment/1_clustering_test')

PATH_CONFIG = {
    # 원천 데이터
    'raw_base': BASE_DIR / 'data' / 'raw' / '23.민간 민원 상담 LLM 사전학습 및 Instruction Tuning 데이터' / '3.개방데이터' / '1.데이터',
    # 가공 데이터
    'processed': BASE_DIR / 'data' / 'processed',
    # 중간 산출물
    'checkpoints': BASE_DIR / 'checkpoints',
    'embeddings': BASE_DIR / 'embeddings',
    'vector_db': BASE_DIR / 'vector_db',
    # 스크립트/노트북
    'scripts': BASE_DIR / 'scripts',
    'notebooks': BASE_DIR / 'notebooks',
}

PATH_CONFIG['train_source'] = PATH_CONFIG['raw_base'] / 'Training' / '01.원천데이터'
PATH_CONFIG['val_source'] = PATH_CONFIG['raw_base'] / 'Validation' / '01.원천데이터'

# Pre-Init 산출물
PATH_CONFIG['pre_init_output'] = PATH_CONFIG['processed'] / 'pre_init_intents.parquet'
PATH_CONFIG['pre_init_ckpt_dir'] = PATH_CONFIG['checkpoints'] / 'pre_init'

# 체크포인트 디렉토리 생성
PATH_CONFIG['pre_init_ckpt_dir'].mkdir(parents=True, exist_ok=True)

# 경로 존재 확인
for key in ['raw_base', 'processed', 'checkpoints', 'train_source', 'val_source']:
    path = PATH_CONFIG[key]
    exists = path.exists()
    marker = '✓' if exists else '✗'
    print(f'  {marker} {key}: {path}')

  ✓ raw_base: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/raw/23.민간 민원 상담 LLM 사전학습 및 Instruction Tuning 데이터/3.개방데이터/1.데이터
  ✓ processed: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed
  ✓ checkpoints: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints
  ✓ train_source: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/raw/23.민간 민원 상담 LLM 사전학습 및 Instruction Tuning 데이터/3.개방데이터/1.데이터/Training/01.원천데이터
  ✓ val_source: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/raw/23.민간 민원 상담 LLM 사전학습 및 Instruction Tuning 데이터/3.개방데이터/1.데이터/Validation/01.원천데이터


## 0-2. 하이퍼파라미터 설정
Pre-Init 단계의 모든 설정값을 하나의 config dict로 관리한다.

In [3]:
# === 하이퍼파라미터 ===
CONFIG = {
    # Gemini API 설정
    'gemini_model': 'gemini-2.5-flash-lite',
    'gemini_temperature': 0.2,          # 낮은 temperature로 일관된 출력
    'gemini_max_output_tokens': 1024,    # 의도 분리 결과 JSON 최대 길이
    'gemini_top_p': 0.95,

    # 배치 처리 설정
    'preinit_batch_size': 100,           # 체크포인트 저장 단위
    'preinit_rpm_limit': 250,            # Tier 1 기준 보수적 RPM (300 중 여유분)
    'preinit_retry_max': 3,              # API 호출 실패 시 재시도 횟수
    'preinit_retry_delay': 5,            # 재시도 간 대기 시간 (초)
    'preinit_rate_limit_delay': 0.25,    # 호출 간 최소 대기 시간 (초) → ~240 RPM

    # 비용 관련
    'budget_total_krw': 50000,
    'exchange_rate': 1450.0,
    'preinit_input_price_per_1m': 0.10,  # USD
    'preinit_output_price_per_1m': 0.40, # USD
}

print('하이퍼파라미터 설정 완료:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

하이퍼파라미터 설정 완료:
  gemini_model: gemini-2.5-flash-lite
  gemini_temperature: 0.2
  gemini_max_output_tokens: 1024
  gemini_top_p: 0.95
  preinit_batch_size: 100
  preinit_rpm_limit: 250
  preinit_retry_max: 3
  preinit_retry_delay: 5
  preinit_rate_limit_delay: 0.25
  budget_total_krw: 50000
  exchange_rate: 1450.0
  preinit_input_price_per_1m: 0.1
  preinit_output_price_per_1m: 0.4


## 0-3. Gemini API 클라이언트 초기화
Colab 보안 비밀에서 API 키를 로드하고 Gemini 클라이언트를 생성한다.

In [4]:
# === Gemini API 초기화 ===
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# 연결 테스트
test_response = client.models.generate_content(
    model=CONFIG['gemini_model'],
    contents='테스트입니다. "ok"라고만 답하세요.',
    config=types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=10,
    ),
)
print(f'Gemini API 연결 성공: {test_response.text}')

Gemini API 연결 성공: ok


## 0-4. 유틸리티 함수 정의
zip 파일 내 JSON 파싱, NFC 정규화, API 호출 래퍼 등 공통 함수

In [5]:
# === 유틸리티 함수 ===

def nfc(text: str) -> str:
    """macOS NFD → NFC 정규화. 한글 파일명/텍스트 인코딩 통일."""
    return unicodedata.normalize('NFC', text)


def load_json_from_zip(zip_path: Path) -> list:
    """
    zip 파일 내 모든 JSON 파일을 읽어 레코드 리스트로 반환한다.
    파일명과 텍스트 내용에 NFC 정규화를 적용한다.
    """
    records = []
    with zipfile.ZipFile(zip_path, 'r') as zf:
        json_files = [
            f for f in zf.namelist()
            if nfc(f).endswith('.json') and '__MACOSX' not in f
        ]
        for jf in tqdm(json_files, desc=f'{zip_path.name}', leave=False):
            with zf.open(jf) as fp:
                raw = fp.read().decode('utf-8')
                raw = nfc(raw)
                data = json.loads(raw)
                if isinstance(data, list):
                    records.extend(data)
                else:
                    records.append(data)
    return records


def call_gemini_with_retry(
    prompt: str,
    system_instruction: str,
    config: dict = CONFIG,
) -> str:
    """
    Gemini API 호출 래퍼. 재시도 로직과 rate limit 대기를 포함한다.
    반환: 모델 응답 텍스트 (실패 시 빈 문자열)
    """
    for attempt in range(config['preinit_retry_max']):
        try:
            response = client.models.generate_content(
                model=config['gemini_model'],
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=config['gemini_temperature'],
                    max_output_tokens=config['gemini_max_output_tokens'],
                    top_p=config['gemini_top_p'],
                    response_mime_type='application/json',
                ),
            )
            time.sleep(config['preinit_rate_limit_delay'])
            return response.text

        except Exception as e:
            error_str = str(e)
            if '429' in error_str or 'RESOURCE_EXHAUSTED' in error_str:
                wait = config['preinit_retry_delay'] * (attempt + 1) * 2
                print(f'  [Rate limit] {wait}초 대기 후 재시도 ({attempt+1}/{config["preinit_retry_max"]})')
                time.sleep(wait)
            else:
                wait = config['preinit_retry_delay'] * (attempt + 1)
                print(f'  [Error] {error_str[:100]}... {wait}초 후 재시도 ({attempt+1}/{config["preinit_retry_max"]})')
                time.sleep(wait)

    print(f'  [FAIL] {config["preinit_retry_max"]}회 재시도 실패')
    return ''


def parse_intent_json(response_text: str, source_id: str) -> list:
    """
    Gemini 응답 JSON을 파싱하여 의도 리스트를 반환한다.
    파싱 실패 시 빈 리스트 반환.
    """
    if not response_text:
        return []
    try:
        data = json.loads(response_text)
        intents = data if isinstance(data, list) else data.get('intents', [])
        return intents
    except json.JSONDecodeError as e:
        print(f'  [JSON 파싱 실패] source_id={source_id}: {str(e)[:80]}')
        return []


print('유틸리티 함수 정의 완료')

유틸리티 함수 정의 완료


## 1. 원천데이터 로드
EDA에서 저장한 체크포인트(`eda_source_df.parquet`)가 있으면 활용하여 메타데이터를 참조하고, zip에서 원문 `consulting_content`를 로드한다.

**체크포인트**: `checkpoints/pre_init_source_records.parquet`

In [6]:
# === 원천데이터 로드 (content 포함) ===

def load_source_with_content() -> pd.DataFrame:
    """
    원천데이터를 consulting_content 원문 포함하여 DataFrame으로 로드한다.
    체크포인트가 존재하면 로드.
    """
    ckpt_path = PATH_CONFIG['checkpoints'] / 'pre_init_source_records.parquet'

    if ckpt_path.exists():
        print(f'체크포인트 로드: {ckpt_path}')
        df = pd.read_parquet(ckpt_path)
        print(f'총 {len(df):,}건 로드')
        return df

    source_zips = {
        'train': sorted(PATH_CONFIG['train_source'].glob('*.zip')),
        'val': sorted(PATH_CONFIG['val_source'].glob('*.zip')),
    }

    all_rows = []
    for split, zips in source_zips.items():
        print(f'\n--- {split.upper()} ---')
        for zp in zips:
            print(f'  로드 중: {zp.name}')
            records = load_json_from_zip(zp)
            for r in records:
                all_rows.append({
                    'source_id': r.get('source_id', ''),
                    'source': nfc(r.get('source', '')),
                    'consulting_category': nfc(r.get('consulting_category', '')),
                    'consulting_turns': int(r.get('consulting_turns', 0)),
                    'consulting_content': nfc(r.get('consulting_content', '')),
                    'split': split,
                })
            print(f'    → {len(records):,}건')

    df = pd.DataFrame(all_rows)
    df.to_parquet(ckpt_path, index=False)
    print(f'\n체크포인트 저장: {ckpt_path}')
    print(f'총 {len(df):,}건')
    return df


source_df = load_source_with_content()
source_df.head(3)


--- TRAIN ---
  로드 중: TS_액티벤처.zip


TS_액티벤처.zip:   0%|          | 0/800 [00:00<?, ?it/s]

    → 800건
  로드 중: TS_엘지유플러스.zip


TS_엘지유플러스.zip:   0%|          | 0/3216 [00:00<?, ?it/s]

    → 3,216건
  로드 중: TS_하나카드.zip


...

VS_엘지유플러스.zip:   0%|          | 0/384 [00:00<?, ?it/s]

    → 384건
  로드 중: VS_하나카드.zip


VS_하나카드.zip:   0%|          | 0/776 [00:00<?, ?it/s]

    → 776건

체크포인트 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints/pre_init_source_records.parquet
총 11,033건


,source_id,source,consulting_category,consulting_turns,consulting_content,split
0,90100,액티벤처,상품예약 및 결제,8,"고객: 안녕하세요, ▲▲월 ▲▲일 출발로 5일 일정으로 2명 예약 가능한지 궁금해요...",train
1,90010,액티벤처,상품예약 및 결제,10,"고객: 안녕하세요. 사파린마린파크 드래곤패키지 성인 2명, 아이 2명 구매하고 싶은...",train
2,90102,액티벤처,상품예약 및 결제,10,"고객: 안녕하세요. 내년 ▲▲월 ▲▲일 결혼인데, ▲▲일에 출발하고 싶어요. 로얄 ...",train


## 2. 의도 분리·요약 프롬프트 정의
Gemini에게 상담 대화를 읽고 고객의 각 의도를 식별·요약하도록 하는 시스템 프롬프트를 정의한다.

**핵심 설계**:
- 한 통화에 여러 의도가 있으면 각각 독립 항목으로 분리
- 단일 의도 상담도 동일한 포맷으로 출력 (intents 배열에 1개)
- 출력: JSON 배열 `{"intents": [{"intent_summary": ..., "relevant_utterances": [...]}, ...]}`
- `relevant_utterances`는 해당 의도와 관련된 핵심 고객 발화를 원문에서 추출

In [7]:
# === 프롬프트 정의 ===

SYSTEM_INSTRUCTION = """당신은 고객 상담 대화를 분석하는 전문가입니다.

## 작업
주어진 상담사-고객 대화를 읽고, 고객이 표현한 **개별 의도(intent)**를 식별하여 구조화된 JSON으로 출력하세요.

## 규칙
1. 고객이 하나의 대화에서 여러 주제를 문의한 경우, 각 주제를 별도의 의도로 분리하세요.
2. 단일 주제만 있는 경우에도 동일한 JSON 구조를 유지하세요 (intents 배열에 1개).
3. `intent_summary`는 **"행위 — 구체적 대상/맥락"** 형식으로 작성하세요.
   - 좋은 예: "확인 요청 — 사전예약 케이스 배송 상태", "문의 — 삼성 케어 플러스 가입용 매장 코드"
   - 나쁜 예: "배송 문의", "기타 문의" (너무 모호함)
4. `relevant_utterances`에는 해당 의도를 뒷받침하는 핵심 고객 발화를 원문 그대로 1~3개 추출하세요.
5. 상담사의 인사말, 본인 확인, 마무리 인사 등 실질적 의도가 아닌 부분은 무시하세요.
6. 마스킹 토큰(<NAME>, <DATE>, ▲▲▲ 등)은 그대로 유지하세요.

## 출력 형식 (JSON)
{
  "intents": [
    {
      "intent_summary": "행위 — 구체적 대상/맥락",
      "relevant_utterances": ["고객 발화 원문1", "고객 발화 원문2"]
    }
  ]
}"""


def build_user_prompt(consulting_content: str) -> str:
    """사용자 프롬프트를 구성한다."""
    return f"""다음 상담 대화를 분석하여 고객의 의도를 분리하고 요약하세요.

<consulting_content>
{consulting_content}
</consulting_content>"""


print('프롬프트 정의 완료')
print(f'시스템 프롬프트 길이: {len(SYSTEM_INSTRUCTION)}자')

프롬프트 정의 완료
시스템 프롬프트 길이: 687자


## 2-1. 프롬프트 검증 (샘플 테스트)
실제 상담 데이터 3건으로 프롬프트가 의도대로 작동하는지 확인한다. 비용: ~0.01 USD 미만.

In [8]:
# === 프롬프트 검증 ===

def test_prompt(df: pd.DataFrame, n_samples: int = 3) -> None:
    """
    랜덤 샘플 n건에 대해 프롬프트를 테스트하고 결과를 출력한다.
    """
    samples = df.sample(n=n_samples, random_state=42)

    for idx, row in samples.iterrows():
        print(f'\n{"="*60}')
        print(f'source_id: {row["source_id"]} | source: {row["source"]} | category: {row["consulting_category"]}')
        print(f'{"="*60}')

        prompt = build_user_prompt(row['consulting_content'])
        response_text = call_gemini_with_retry(prompt, SYSTEM_INSTRUCTION)

        if response_text:
            intents = parse_intent_json(response_text, row['source_id'])
            print(f'의도 수: {len(intents)}')
            for i, intent in enumerate(intents):
                print(f'  [{i+1}] {intent.get("intent_summary", "N/A")}')
                for u in intent.get('relevant_utterances', [])[:2]:
                    print(f'      → {u[:80]}...' if len(u) > 80 else f'      → {u}')
        else:
            print('  [FAIL] 응답 없음')


test_prompt(source_df, n_samples=3)


source_id: 22002 | source: 하나카드 | category: 긴급 배송 신청 
의도 수: 2
  [1] 문의 — 체크 카드 배송 및 수령 방법
      → 제가 ▲▲ 체크 카드 신청을 했는데요.
      → 이게 배송을 미리 받을 수 있거나 은행 가서 발급받으면 되나요?
  [2] 요청 — 체크 카드 긴급 배송 신청
      → 아 그러면 일단 긴급 배송 신청 좀 해 주세요.

source_id: 40892 | source: 엘지유플러스 | category: 소액 결제
의도 수: 2
  [1] 요청 — 소액결제 및 구글플레이 결제 차단
      → 예, 저기 소액결제 차단 좀 하려구요.
      → 근데 이거 차단하면 구글플레이 이런 것도 다 차단되는 거예요?
  [2] 확인 요청 — 아들 명의 확인 및 결제 차단
      → 그거 저희 아들 누구 명의로 돼 있나 한번 좀 확인 좀 해 주실래요?
      → 아 그렇다면 차단해 주세요. 구글 최소로 무조건 다 차단해 주세요.

source_id: 200118 | source: 하나카드 | category: 선결제/즉시출금
의도 수: 1
  [1] 결제 요청 — 당월 카드 대금
      → 아 예, 아 저 카드 결제하려고 하는데요?
      → 네 네, 그 결제해 주세요.


## 3. 전체 데이터 의도 분리·요약 실행
전체 11,033건을 처리한다. 100건 단위로 체크포인트를 저장하여 세션이 끊어져도 재개할 수 있다.

**재개 로직**: `checkpoints/pre_init/` 디렉토리에서 이미 처리된 `source_id`를 수집하고, 미처리 건만 처리한다.

**예상 소요**: Tier 1 기준 ~1.2시간, 비용 ~₩5,200

In [10]:
# === 전체 처리 ===

def get_processed_source_ids() -> set:
    """
    체크포인트 디렉토리에서 이미 처리된 source_id 집합을 반환한다.
    """
    ckpt_dir = PATH_CONFIG['pre_init_ckpt_dir']
    processed_ids = set()
    for f in ckpt_dir.glob('pre_init_batch_*.parquet'):
        try:
            batch_df = pd.read_parquet(f)
            processed_ids.update(batch_df['source_id'].unique())
        except Exception as e:
            print(f'  [경고] 체크포인트 읽기 실패: {f.name}: {e}')
    return processed_ids


def process_single_record(row: pd.Series) -> list:
    """
    단일 상담 레코드를 처리하여 의도 분리 결과 dict 리스트를 반환한다.
    각 dict는 하나의 의도에 대응한다.
    """
    prompt = build_user_prompt(row['consulting_content'])
    response_text = call_gemini_with_retry(prompt, SYSTEM_INSTRUCTION)
    intents = parse_intent_json(response_text, row['source_id'])

    results = []
    if not intents:
        # 파싱 실패 시 원본 카테고리를 fallback으로 사용
        results.append({
            'source_id': row['source_id'],
            'source': row['source'],
            'consulting_category': row['consulting_category'],
            'split': row['split'],
            'intent_index': 0,
            'intent_summary': f'[FALLBACK] {row["consulting_category"]}',
            'relevant_utterances': '[]',
            'n_intents': 1,
            'is_fallback': True,
            'raw_response': response_text[:500] if response_text else '',
        })
    else:
        for i, intent in enumerate(intents):
            results.append({
                'source_id': row['source_id'],
                'source': row['source'],
                'consulting_category': row['consulting_category'],
                'split': row['split'],
                'intent_index': i,
                'intent_summary': intent.get('intent_summary', ''),
                'relevant_utterances': json.dumps(
                    intent.get('relevant_utterances', []),
                    ensure_ascii=False,
                ),
                'n_intents': len(intents),
                'is_fallback': False,
                'raw_response': '',
            })
    return results


def run_pre_init(df: pd.DataFrame) -> pd.DataFrame:
    """
    전체 데이터에 대해 의도 분리·요약을 실행한다.
    이미 처리된 건은 건너뛰고, 100건 단위로 체크포인트를 저장한다.
    """
    # 이미 처리된 ID 확인
    processed_ids = get_processed_source_ids()
    remaining = df[~df['source_id'].isin(processed_ids)].copy()

    print(f'전체: {len(df):,}건 | 처리 완료: {len(processed_ids):,}건 | 남은 건수: {len(remaining):,}건')

    if len(remaining) == 0:
        print('모든 건이 이미 처리됨. 체크포인트에서 결과를 병합합니다.')
        return merge_checkpoints()

    # 배치 처리
    batch_results = []
    batch_counter = len(list(PATH_CONFIG['pre_init_ckpt_dir'].glob('pre_init_batch_*.parquet')))
    fail_count = 0

    pbar = tqdm(remaining.iterrows(), total=len(remaining), desc='Pre-Init 처리')
    for idx, row in pbar:
        results = process_single_record(row)
        batch_results.extend(results)

        if results and results[0]['is_fallback']:
            fail_count += 1

        pbar.set_postfix({
            'intents': sum(r['n_intents'] for r in results),
            'fails': fail_count,
            'batch': batch_counter,
        })

        # 배치 체크포인트 저장
        if len(batch_results) >= CONFIG['preinit_batch_size']:
            batch_df = pd.DataFrame(batch_results)
            ckpt_path = PATH_CONFIG['pre_init_ckpt_dir'] / f'pre_init_batch_{batch_counter:04d}.parquet'
            batch_df.to_parquet(ckpt_path, index=False)
            batch_counter += 1
            batch_results = []

    # 남은 배치 저장
    if batch_results:
        batch_df = pd.DataFrame(batch_results)
        ckpt_path = PATH_CONFIG['pre_init_ckpt_dir'] / f'pre_init_batch_{batch_counter:04d}.parquet'
        batch_df.to_parquet(ckpt_path, index=False)

    print(f'\n처리 완료. 실패(fallback): {fail_count}건')
    return merge_checkpoints()


def merge_checkpoints() -> pd.DataFrame:
    """
    체크포인트 파일들을 병합하여 최종 DataFrame을 반환하고 저장한다.
    """
    ckpt_dir = PATH_CONFIG['pre_init_ckpt_dir']
    ckpt_files = sorted(ckpt_dir.glob('pre_init_batch_*.parquet'))

    if not ckpt_files:
        print('체크포인트 파일이 없습니다.')
        return pd.DataFrame()

    dfs = [pd.read_parquet(f) for f in ckpt_files]
    combined = pd.concat(dfs, ignore_index=True)

    # 중복 제거 (source_id + intent_index 기준)
    combined = combined.drop_duplicates(subset=['source_id', 'intent_index'], keep='last')

    # 최종 저장
    output_path = PATH_CONFIG['pre_init_output']
    combined.to_parquet(output_path, index=False)
    print(f'최종 결과 저장: {output_path}')
    print(f'총 의도 레코드: {len(combined):,}건 (원본 {combined["source_id"].nunique():,}건에서 분리)')

    return combined


# 실행
intent_df = run_pre_init(source_df)

전체: 11,033건 | 처리 완료: 174건 | 남은 건수: 10,859건


Pre-Init 처리:   0%|          | 0/10859 [00:00<?, ?it/s]

  [Error] 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman... 5초 후 재시도 (1/3)
  [Error] 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman... 5초 후 재시도 (1/3)
  [Error] 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman... 5초 후 재시도 (1/3)
  [Error] 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman... 5초 후 재시도 (1/3)
  [Error] 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman... 5초 후 재시도 (1/3)
  [Error] 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman... 10초 후 재시도 (2/3)
  [Error] 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman... 5초 후 재시도 (1/3)
  [Error] 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experienc

## 4. 결과 검증 및 통계
의도 분리 결과의 품질과 분포를 확인한다.

In [11]:
# === 결과 검증 ===

def validate_pre_init_results(df: pd.DataFrame) -> dict:
    """
    Pre-Init 결과를 검증하고 통계를 출력한다.
    """
    stats = {}

    # 기본 통계
    total_intents = len(df)
    total_sources = df['source_id'].nunique()
    expansion_rate = total_intents / total_sources

    print(f'=== Pre-Init 결과 ===')
    print(f'  원본 상담: {total_sources:,}건')
    print(f'  분리된 의도: {total_intents:,}건')
    print(f'  확장 비율: {expansion_rate:.2f}x')
    stats['total_intents'] = total_intents
    stats['total_sources'] = total_sources
    stats['expansion_rate'] = round(expansion_rate, 2)

    # Fallback 비율
    fallback_count = df['is_fallback'].sum()
    print(f'\n  Fallback(파싱실패): {fallback_count:,}건 ({fallback_count/total_intents*100:.1f}%)')
    stats['fallback_count'] = int(fallback_count)

    # 의도 수 분포 (원본 상담당)
    print(f'\n--- 상담당 의도 수 분포 ---')
    intent_counts = df.groupby('source_id')['intent_index'].max() + 1
    print(intent_counts.value_counts().sort_index().to_string())
    stats['intent_count_dist'] = intent_counts.value_counts().sort_index().to_dict()

    # 출처별 통계
    print(f'\n--- 출처별 ---')
    for src in sorted(df['source'].unique()):
        sub = df[df['source'] == src]
        src_sources = sub['source_id'].nunique()
        print(f'  {src}: {len(sub):,} 의도 / {src_sources:,} 상담 = {len(sub)/src_sources:.2f}x')

    # intent_summary 길이 분포
    non_fallback = df[~df['is_fallback']]
    if len(non_fallback) > 0:
        summary_len = non_fallback['intent_summary'].str.len()
        print(f'\n--- intent_summary 길이 (글자 수) ---')
        print(summary_len.describe().to_string())
        stats['summary_len'] = summary_len.describe().to_dict()

    # 샘플 출력
    print(f'\n--- 샘플 (복수 의도 상담) ---')
    multi = df[df['n_intents'] > 1]
    if len(multi) > 0:
        sample_sid = multi.sample(1, random_state=42)['source_id'].iloc[0]
        sample_rows = df[df['source_id'] == sample_sid]
        print(f'source_id: {sample_sid} | category: {sample_rows.iloc[0]["consulting_category"]} | 의도 수: {len(sample_rows)}')
        for _, r in sample_rows.iterrows():
            print(f'  [{r["intent_index"]}] {r["intent_summary"]}')

    return stats


pre_init_stats = validate_pre_init_results(intent_df)

=== Pre-Init 결과 ===
  원본 상담: 11,033건
  분리된 의도: 34,025건
  확장 비율: 3.08x

  Fallback(파싱실패): 11건 (0.0%)

--- 상담당 의도 수 분포 ---
intent_index
1      880
2     3382
3     3242
4     1950
5      918
6      449
7      151
8       43
9       14
10       2
11       1
13       1

--- 출처별 ---
  액티벤처: 4,209 의도 / 900 상담 = 4.68x
  엘지유플러스: 11,662 의도 / 3,600 상담 = 3.24x
  하나카드: 18,154 의도 / 6,533 상담 = 2.78x

--- intent_summary 길이 (글자 수) ---
count    34014.000000
mean        20.642765
std          5.823940
min          7.000000
25%         16.000000
50%         20.000000
75%         24.000000
max         66.000000

--- 샘플 (복수 의도 상담) ---
source_id: 42772 | category: 납부 방법 변경 | 의도 수: 5
  [0] 확인 요청 — 미납금 존재 여부
  [1] 문의 — 카드 결제 가능 여부 및 변경
  [2] 문의 — 신용카드/체크카드 구분
  [3] 요청 — 즉시 결제
  [4] 문의 — 카드 할부


## 5. 라벨링 요약 데이터와 품질 비교 (참고)
라벨링 `_요약` 데이터의 output과 Gemini 의도 요약을 동일 `source_id`에서 비교하여, 핵심 정보 누락 여부를 정성적으로 확인한다.

In [12]:
# === 요약 품질 비교 (정성 확인용) ===

def compare_with_label_summary(intent_df: pd.DataFrame, n_samples: int = 3) -> None:
    """
    라벨링 요약 데이터와 Gemini 의도 분리 결과를 source_id 기준으로 비교한다.
    라벨링 데이터는 요약 zip에서 로드한다.
    """
    # 라벨링 요약 데이터 로드 (Training 엘지유플러스 요약만 샘플용)
    label_dir = PATH_CONFIG['raw_base'] / 'Training' / '02.라벨링데이터'
    summary_zips = [z for z in label_dir.glob('*요약*.zip')]

    if not summary_zips:
        print('요약 라벨링 zip을 찾을 수 없습니다.')
        return

    # 첫 번째 요약 zip에서 샘플 로드
    zp = summary_zips[0]
    print(f'라벨링 요약 로드: {zp.name}')
    label_records = load_json_from_zip(zp)

    # source_id → 라벨 요약 매핑
    label_map = {}
    for r in label_records:
        sid = r.get('source_id', '')
        for inst_group in r.get('instructions', []):
            for d in inst_group.get('data', []):
                if d.get('task') == '요약':
                    label_map[sid] = d.get('output', '')
                    break

    # Gemini 결과와 매칭되는 source_id 찾기
    common_ids = set(intent_df['source_id'].unique()) & set(label_map.keys())
    print(f'매칭 가능한 source_id: {len(common_ids)}건')

    if not common_ids:
        print('매칭되는 source_id가 없습니다.')
        return

    sample_ids = list(common_ids)[:n_samples]
    for sid in sample_ids:
        print(f'\n{"="*60}')
        print(f'source_id: {sid}')
        print(f'{"="*60}')

        # 라벨 요약
        print(f'[라벨 요약]')
        print(f'  {label_map[sid]}')

        # Gemini 의도 분리
        gemini_rows = intent_df[intent_df['source_id'] == sid]
        print(f'\n[Gemini 의도 분리] ({len(gemini_rows)}개 의도)')
        for _, r in gemini_rows.iterrows():
            print(f'  [{r["intent_index"]}] {r["intent_summary"]}')


compare_with_label_summary(intent_df, n_samples=3)

요약 라벨링 zip을 찾을 수 없습니다.


## 6. 비용 정산
실제 사용한 토큰 수를 기반으로 비용을 정산한다. (추정치와 비교)

In [13]:
# === 비용 정산 ===

def estimate_actual_cost(df: pd.DataFrame) -> dict:
    """
    처리 결과로부터 실제 비용을 추정한다.
    정확한 토큰 카운트는 API 대시보드에서 확인 필요.
    여기서는 글자 수 기반 추정치를 계산한다.
    """
    total_sources = df['source_id'].nunique()
    total_intents = len(df)

    # 입력 토큰 추정: EDA 결과 기준
    est_input_tokens = total_sources * (1729 + 500)  # content avg + prompt

    # 출력 토큰 추정: intent_summary + relevant_utterances 기반
    non_fallback = df[~df['is_fallback']]
    if len(non_fallback) > 0:
        avg_output_chars = (
            non_fallback['intent_summary'].str.len().mean()
            + non_fallback['relevant_utterances'].str.len().mean()
        )
        est_output_tokens = int(total_sources * avg_output_chars * 1.3)
    else:
        est_output_tokens = total_sources * 250

    input_cost = (est_input_tokens / 1_000_000) * CONFIG['preinit_input_price_per_1m']
    output_cost = (est_output_tokens / 1_000_000) * CONFIG['preinit_output_price_per_1m']
    total_usd = input_cost + output_cost
    total_krw = total_usd * CONFIG['exchange_rate']

    print(f'=== Pre-Init 비용 추정 ===')
    print(f'  입력 토큰: {est_input_tokens:,.0f} → ${input_cost:.4f}')
    print(f'  출력 토큰: {est_output_tokens:,.0f} → ${output_cost:.4f}')
    print(f'  합계: ${total_usd:.4f} (≈ ₩{total_krw:,.0f})')
    print(f'  잔여 예산: ₩{CONFIG["budget_total_krw"] - total_krw:,.0f}')
    print(f'\n※ 정확한 비용은 Google AI Studio 대시보드에서 확인하세요.')

    return {
        'est_input_tokens': est_input_tokens,
        'est_output_tokens': est_output_tokens,
        'total_usd': round(total_usd, 4),
        'total_krw': round(total_krw, 0),
    }


cost_result = estimate_actual_cost(intent_df)

=== Pre-Init 비용 추정 ===
  입력 토큰: 24,592,557 → $2.4593
  출력 토큰: 1,465,144 → $0.5861
  합계: $3.0453 (≈ ₩4,416)
  잔여 예산: ₩45,584

※ 정확한 비용은 Google AI Studio 대시보드에서 확인하세요.


## 7. 종료
세션을 자동으로 종료하고, 자원을 반환한다.

In [14]:
from google.colab import runtime
runtime.unassign()